#### 4. Build a full SCD Type 2 table: implement the MERGE that closes out old records (setting end_date and is_current) and inserts new versions when a tracked column changes.

In [0]:
%sql
create table cyntexa_dev.silver.products (
    product_id int, 
    name string,
    category string,
    price double
)

In [0]:
%sql
INSERT INTO cyntexa_dev.silver.products
VALUES
(1, 'Product1', 'Category1', 100.00),
(2, 'Product2', 'Category2', 150.00),
(3, 'Product3', 'Category1', 200.00),
(4, 'Product4', 'Category2', 250.00),
(5, 'Product5', 'Category1', 300.00),
(6, 'Product6', 'Category2', 350.00),
(7, 'Product7', 'Category1', 400.00),
(8, 'Product8', 'Category2', 450.00),
(9, 'Product9', 'Category1', 500.00),
(10, 'Product10', 'Category2', 550.00);

In [0]:
%sql
create table cyntexa_dev.silver.products_scd2 as 
select *,
cast(null as date) as effective_date,
cast(null as date) as end_date,
cast(null as int) as version,
cast(null  as boolean) as is_current
from cyntexa_dev.silver.products
where 1 = 0

In [0]:
%sql
merge into cyntexa_dev.silver.products_scd2 t
using cyntexa_dev.silver.products s 
on t.product_id = s.product_id and t.is_current = true
when matched and (not(t.name <=> s.name) or
not(t.category <=> s.category) or
not(t.price <=> s.price))
then update set
t.end_date = current_date(),
t.is_current = false
when not matched then
insert (product_id, name, category, price, effective_date, end_date, version, is_current)
values (s.product_id, s.name, s.category, s.price, current_date(), null, 1, true);

insert into cyntexa_dev.silver.products_scd2
select
s.product_id,
s.name,
s.category,
s.price,
current_date() as effective_date,
null as end_date,
max(t.version) + 1 as version,
true as is_current
from cyntexa_dev.silver.products s
inner join cyntexa_dev.silver.products_scd2 t 
on t.product_id = s.product_id and t.end_date = current_date()
where not(t.name <=> s.name) or
not(t.category <=> s.category) or
not(t.price <=> s.price)
group by s.product_id,
s.name,
s.category,
s.price

#### 5. Query the SCD Type 2 table to answer a point-in-time question, e.g. 'what was this customer's address as of March 1st?'

In [0]:
%sql
select * from cyntexa_dev.silver.products_scd2
where effective_date <= current_date()

#### 6. Compare the estimated DBU cost of running a job on all-purpose vs. job compute, and recommend which Cyntexa should use for its nightly pipeline.


## DBU Cost Comparison: All-Purpose vs. Job Compute

### Cost Analysis

**All-Purpose Compute:**
- **DBU Rate:** 2-4x higher than Job Compute (varies by cloud and instance type)
- **Typical Rate:** ~$0.40-0.75 per DBU (AWS, depending on instance)
- **Use Case:** Interactive development, ad-hoc analysis, notebooks
- **Always-on capability:** Can keep clusters running for immediate use
- **Idle time:** You pay for the entire cluster uptime, even when idle

**Job Compute:**
- **DBU Rate:** Significantly lower, optimized for production workloads
- **Typical Rate:** ~$0.10-0.22 per DBU (AWS, depending on instance)
- **Cost Savings:** **50-75% lower** than All-Purpose Compute
- **Use Case:** Scheduled jobs, automated pipelines, production ETL
- **Ephemeral:** Clusters start for the job and terminate when complete
- **No idle cost:** You only pay for actual job execution time

### Cost Example

For a nightly pipeline that runs 2 hours per night:
- **All-Purpose (at $0.50/DBU):** If using 10 DBUs/hour = 10 × 2 × $0.50 = **$10/night** = **$300/month**
- **Job Compute (at $0.15/DBU):** Same workload = 10 × 2 × $0.15 = **$3/night** = **$90/month**
- **Savings:** **$210/month (70% reduction)**

### Recommendation for Cyntexa's Nightly Pipeline

**Use Job Compute** for the following reasons:

1. **Cost Efficiency:** 50-75% cost savings compared to All-Purpose
2. **Production Workload:** Nightly pipelines are scheduled, automated jobs—exactly what Job Compute is designed for
3. **No Idle Time:** Job clusters terminate after completion, eliminating idle costs
4. **Optimized Performance:** Job Compute is optimized for throughput and reliability
5. **Best Practice:** Databricks recommends Job Compute for all production pipelines

### Implementation

- Configure your workflow/job in the Databricks Workflows UI
- Select "Job Compute" when configuring the cluster
- Set appropriate cluster sizing based on your SCD Type 2 workload
- Enable autoscaling if workload varies
- Schedule for nightly execution using cron expressions

**Bottom Line:** For Cyntexa's nightly SCD Type 2 pipeline, Job Compute will deliver the same results at a fraction of the cost.
